# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [1]:
# Ваш код

import json

# Загружаем сообщения ЕФРСБ
with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

# Загружаем реестр организаций
with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

# Загружаем приоритетные дела (по строкам, обрезая \n и \r)
with open('priority_cases.txt', 'r', encoding='utf-8') as f:
    priority_case_numbers = [line.strip() for line in f if line.strip()]

print('Количество сообщений:', len(messages))
print('Количество организаций:', len(organizations))
print('Количество приоритетных дел:', len(priority_case_numbers))

Количество сообщений: 54
Количество организаций: 30
Количество приоритетных дел: 10


### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [2]:
# Ваш код

# dict comprehension: пробегаем по списку organizations и строим словарь
org_by_inn = {org['inn']: org for org in organizations}

print('Количество организаций в словаре:', len(org_by_inn))

inn_to_check = '7701234567'
print(f'Организация с ИНН {inn_to_check}:')
print(org_by_inn.get(inn_to_check))

Количество организаций в словаре: 30
Организация с ИНН 7701234567:
{'inn': '7701234567', 'ogrn': '1027700123456', 'name': 'ООО "Альфа-Строй"', 'address': 'г. Москва, ул. Строителей, д. 15', 'region': 'Москва'}


### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [3]:
# Ваш код

linked_count = 0
unlinked_count = 0

for msg in messages:
    inn = msg.get('publisher_inn')
    org = org_by_inn.get(inn)
    if org is not None:
        msg['org_name'] = org.get('name')
        msg['region'] = org.get('region')
        linked_count += 1
    else:
        msg['org_name'] = None
        msg['region'] = None
        unlinked_count += 1

print('Сообщений со связанной организацией:', linked_count)
print('Сообщений без организации:', unlinked_count)

Сообщений со связанной организацией: 51
Сообщений без организации: 3


### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [4]:
# Ваш код

# Множество приоритетных дел
priority_set = set(priority_case_numbers)

# Множество номеров дел из сообщений (фильтруем пустые значения)
message_case_set = set(
    msg['case_number']
    for msg in messages
    if msg.get('case_number')
)

# Пересечение множеств
intersection_cases = priority_set & message_case_set

print('\nНайдено пересечений:', len(intersection_cases))

print('Список пересечений:')
for cn in sorted(intersection_cases):
    print(cn)


Найдено пересечений: 10
Список пересечений:
А60-11111/2025
А60-12345/2025
А60-22222/2025
А60-33333/2025
А60-44444/2025
А60-56789/2025
А60-66666/2025
А60-77777/2025
А60-88888/2025
А60-99999/2025


---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [5]:
# Ваш код

from datetime import datetime

# Словарь для русских названий месяцев
MONTHS_RU = {
    'января': '01',
    'февраля': '02',
    'марта': '03',
    'апреля': '04',
    'мая': '05',
    'июня': '06',
    'июля': '07',
    'августа': '08',
    'сентября': '09',
    'октября': '10',
    'ноября': '11',
    'декабря': '12',
}

def parse_date(date_str):
    
    if not date_str:
        return None

    # Приводим к строке и обрезаем пробелы
    s = str(date_str).strip()

    # Пробуем разные форматы последовательно
    # DD.MM.YYYY
    try:
        return datetime.strptime(s, '%d.%m.%Y')
    except Exception:
        pass

    # YYYY-MM-DD или полный ISO (YYYY-MM-DDTHH:MM:SS)
    # fromisoformat сам умеет понимать разные ISO‑варианты
    try:
        return datetime.fromisoformat(s)
    except Exception:
        pass

    # DD месяца YYYY г.
    try:
        parts = s.split()
        # ожидаем минимум 3 части: день, месяц-словом, год...
        if len(parts) >= 3:
            day = parts[0]
            month_word = parts[1].lower()
            year = parts[2]
            month_num = MONTHS_RU.get(month_word)
            if month_num:
                normalized = f'{day}.{month_num}.{year}'
                return datetime.strptime(normalized, '%d.%m.%Y')
    except Exception:
        pass

    # DD/MM/YYYY HH:MM
    try:
        return datetime.strptime(s, '%d/%m/%Y %H:%M')
    except Exception:
        pass

    # Если ничего не сработало
    return None

# Проверка работы функции
test_dates = [
    '15.01.2025',            # DD.MM.YYYY
    '2025-02-10',            # YYYY-MM-DD
    '20 марта 2025 г.',      # DD месяца YYYY г.
    '15/03/2025 10:30',      # DD/MM/YYYY HH:MM
    '2025-03-15T00:00:00',   # ISO-строка с временем
    None,                    # None
    '',                      # пустая строка
    'ДАТА УТЕРЯНА'           # некорректный формат
]

print('Проверка parse_date:\n')
for d in test_dates:
    result = parse_date(d)
    print(f'Исходное значение: {repr(d)} -> результат: {result}')

Проверка parse_date:

Исходное значение: '15.01.2025' -> результат: 2025-01-15 00:00:00
Исходное значение: '2025-02-10' -> результат: 2025-02-10 00:00:00
Исходное значение: '20 марта 2025 г.' -> результат: 2025-03-20 00:00:00
Исходное значение: '15/03/2025 10:30' -> результат: 2025-03-15 10:30:00
Исходное значение: '2025-03-15T00:00:00' -> результат: 2025-03-15 00:00:00
Исходное значение: None -> результат: None
Исходное значение: '' -> результат: None
Исходное значение: 'ДАТА УТЕРЯНА' -> результат: None


### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [6]:
'''
ВАЖНО!

Функция не проверяет математическую корректность ИНН по контрольным цифрам,
поскольку такая задача поставлена не была

'''

# Ваш код

def validate_message(msg):
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']

    # Проверяем обязательные поля
    for field in required_fields:
        value = msg.get(field)
        if value is None or (isinstance(value, str) and not value.strip()):
            errors.append({
                'type': 'KeyError',
                'message': f"Отсутствует обязательное поле '{field}'"
            })

    # Проверяем ИНН
    publisher_inn = msg.get('publisher_inn')
    cleaned_inn = None
    if publisher_inn is not None and str(publisher_inn).strip():
        cleaned_inn = str(publisher_inn).strip()
        if not (cleaned_inn.isdigit() and len(cleaned_inn) in (10, 12)):
            errors.append({
                'type': 'ValueError',
                'message': f"Некорректный ИНН: {publisher_inn}"
            })

    # Парсим дату через parse_date() из задания 2.1
    date_value = msg.get('date_published')
    parsed_date = None
    if date_value is not None and str(date_value).strip():
        parsed_date = parse_date(date_value)
        if parsed_date is None:
            errors.append({
                'type': 'ValueError',
                'message': f"Не удалось распарсить дату: {date_value}"
            })

    # Проверяем тип сообщения
    msg_type = msg.get('type')
    if msg_type is not None and not (isinstance(msg_type, str) and msg_type.strip()):
        errors.append({
            'type': 'ValueError',
            'message': "Поле 'type' должно быть непустой строкой"
        })

    # Если есть ошибки - сообщение невалидно
    if errors:
        return None, errors

    # Если все хорошо - возвращаем очищенную копию
    valid_msg = dict(msg)
    valid_msg['publisher_inn'] = cleaned_inn
    valid_msg['date_published'] = parsed_date

    return valid_msg, []

# Проверка работы функции
test_msg_ok = {
    'publisher_inn': '7701234567',
    'msg_text': 'Сообщение о введении процедуры наблюдения.',
    'date_published': '15.01.2025',
    'type': 'Введение процедуры',
    'case_number': 'А60-12345/2025'
}

valid_msg, errors = validate_message(test_msg_ok)

print('Валидное сообщение:')
print(valid_msg)
print('Ошибки:')
if errors:
    for e in errors:
        print(f"- [{e['type']}] {e['message']}")
else:
    print('Ошибок нет')


test_msg_bad = {
    'publisher_inn': '7705',
    'msg_text': 'Текст без корректной даты и с плохим ИНН.',
    'date_published': 'ДАТА УТЕРЯНА',
    'type': '',
    'case_number': 'А60-00000/2025'
}

valid_msg2, errors2 = validate_message(test_msg_bad)

print('\nНевалидное сообщение:')
print(valid_msg2)
print('Ошибки:')
if errors2:
    for e in errors2:
        print(f"- [{e['type']}] {e['message']}")
else:
    print('Ошибок нет')

Валидное сообщение:
{'publisher_inn': '7701234567', 'msg_text': 'Сообщение о введении процедуры наблюдения.', 'date_published': datetime.datetime(2025, 1, 15, 0, 0), 'type': 'Введение процедуры', 'case_number': 'А60-12345/2025'}
Ошибки:
Ошибок нет

Невалидное сообщение:
None
Ошибки:
- [KeyError] Отсутствует обязательное поле 'type'
- [ValueError] Некорректный ИНН: 7705
- [ValueError] Не удалось распарсить дату: ДАТА УТЕРЯНА
- [ValueError] Поле 'type' должно быть непустой строкой


### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [7]:
# Ваш код

valid_messages = []
error_records = []

# Статистика по типам ошибок
error_type_stats = {}

# Статистика по текстам ошибок
error_message_stats = {}

for msg in messages:
    valid_msg, errors = validate_message(msg)

    if valid_msg is not None:
        valid_messages.append(valid_msg)
    else:
        error_records.append({
            'message': msg,
            'errors': errors
        })

        for err in errors:
            err_type = err['type']
            err_message = err['message']

            error_type_stats[err_type] = error_type_stats.get(err_type, 0) + 1
            error_message_stats[err_message] = error_message_stats.get(err_message, 0) + 1

print('Всего сообщений:', len(messages))
print('Количество валидных сообщений:', len(valid_messages))
print('Количество сообщений с ошибками:', len(error_records))

print('\nСтатистика ошибок по типам:')
for err_type, count in sorted(error_type_stats.items(), key=lambda x: x[1], reverse=True):
    print(f'{err_type}: {count}')

print('\nСтатистика ошибок по описаниям:')
for err_message, count in sorted(error_message_stats.items(), key=lambda x: x[1], reverse=True):
    print(f'{err_message}: {count}')

print('\nДетализация ошибок:')
for rec in error_records:
    case_number = rec['message'].get('case_number', 'UNKNOWN')
    publisher_inn = rec['message'].get('publisher_inn', 'UNKNOWN')
    errors_str = '; '.join(err['message'] for err in rec['errors'])
    print(f'{case_number} | ИНН {publisher_inn}: {errors_str}')

Всего сообщений: 54
Количество валидных сообщений: 48
Количество сообщений с ошибками: 6

Статистика ошибок по типам:
KeyError: 4
ValueError: 2

Статистика ошибок по описаниям:
Отсутствует обязательное поле 'publisher_inn': 1
Отсутствует обязательное поле 'date_published': 1
Отсутствует обязательное поле 'type': 1
Некорректный ИНН: 7705: 1
Не удалось распарсить дату: ДАТА УТЕРЯНА: 1
Отсутствует обязательное поле 'case_number': 1

Детализация ошибок:
А60-24680/2025 | ИНН : Отсутствует обязательное поле 'publisher_inn'
А60-36925/2025 | ИНН 7701234567: Отсутствует обязательное поле 'date_published'
А60-25814/2025 | ИНН 7702345678: Отсутствует обязательное поле 'type'
А60-12345/2025 | ИНН 7705: Некорректный ИНН: 7705
А60-56789/2025 | ИНН 6658123450: Не удалось распарсить дату: ДАТА УТЕРЯНА
 | ИНН 7703901234: Отсутствует обязательное поле 'case_number'


---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [8]:
#  Ваш код

def extract_amounts(text):
    if not text:
        return []

    result = []
    words = text.split()
    i = 0

    while i < len(words):
        w = words[i]

        if w.endswith('руб.') or 'руб.' in w:
            start = max(0, i - 3)
            fragment_words = words[start:i+1]

            amount_start = None
            for j in range(len(fragment_words)):
                if any(ch.isdigit() for ch in fragment_words[j]):
                    amount_start = j
                    break

            if amount_start is not None:
                amount = " ".join(fragment_words[amount_start:])
                result.append(amount)

            i += 1
            continue

        if w in ('тыс.', 'млн') and i + 1 < len(words) and 'руб.' in words[i+1]:
            
            if i - 1 >= 0:
                amount = f"{words[i-1]} {w} {words[i+1]}"
                result.append(amount)
                i += 2
                continue

        i += 1

    return result

# Проверка работы функции
print(extract_amounts('Долг составляет 15 000 000 руб. АС Свердловской области.'))

print(extract_amounts('Сумма требований: 8 500 тыс. руб. Общая сумма задолженности: 42 млн руб.'))

print(extract_amounts('Сообщение без упоминания денег'))

['15 000 000 руб.']
['500 тыс. руб.', '42 млн руб.']
[]


### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [9]:
# Ваш код

def find_court_mentions(text):
    
    if not text:
        return False
    return 'АС ' in text

# Проверка работы функции
print(find_court_mentions('АС г. Москвы'))
print(find_court_mentions('Судебное заседание назначено'))

True
False


### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [10]:
# Ваш код

def extract_manager_name(text):

    if not text:
        return None

    lower_text = text.lower()
    phrase = 'арбитражный управляющий'
    idx = lower_text.find(phrase)
    if idx == -1:
        return None

    # Берем часть текста после фразы
    start_pos = idx + len(phrase)
    tail = text[start_pos:].strip()

    # Разбиваем на слова и берем первые 3
    parts = tail.split()
    if len(parts) < 3:
        return None

    fio = ' '.join(parts[:3])

    # Убираем точку и пробелы в конце, если есть
    fio = fio.rstrip(' .,')

    return fio

# Проверка работы функции
# Корректное ФИО
text1 = 'Сообщение о введении процедуры. Арбитражный управляющий Иванов Иван Иванович. Долг составляет 15 000 000 руб.'
print(extract_manager_name(text1))  # Иванов Иван Иванович

# Другое корректное ФИО
text2 = 'Арбитражный управляющий Петров Пётр Петрович уведомил кредиторов о собрании.'
print(extract_manager_name(text2))  # Петров Пётр Петрович

# В тексте нет арбитражного управляющего
text3 = "Сообщение о введении процедуры наблюдения без указания арбитражного управляющего."
print(extract_manager_name(text3))  # None

# Недостаточно слов после фразы слов "арбитражный управляющий"
text4 = 'Арбитражный управляющий Иванов.'
print(extract_manager_name(text4))  # None (меньше 3 слов после фразы)

# Пустой текст
text5 = ''
print(extract_manager_name(text5))  # None

Иванов Иван Иванович
Петров Пётр Петрович
None
None
None


### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [11]:
# Ваш код

for msg in valid_messages:
    text = msg.get('msg_text', '')
    msg['amounts'] = extract_amounts(text)
    msg['has_court_mention'] = find_court_mentions(text)
    msg['manager_name'] = extract_manager_name(text)

print('Всего валидных сообщений:', len(valid_messages))

# Проверяем наличие новых полей у всех валидных сообщений
missing_fields = {
    'amounts': 0,
    'has_court_mention': 0,
    'manager_name': 0,
}

for msg in valid_messages:
    for field in missing_fields.keys():
        if field not in msg:
            missing_fields[field] += 1

print('\nКоличество сообщений, в которых отсутствуют новые поля:')
for field, cnt in missing_fields.items():
    print(f'{field}: {cnt}')

# Выведем первые три сообщения после обогащения
print('\nПервые 3 сообщения после обогащения:\n')
for msg in valid_messages[:3]:
    print('case_number:', msg.get('case_number'))
    print('org_name:', msg.get('org_name'))
    print('amounts:', msg.get('amounts'))
    print('has_court_mention:', msg.get('has_court_mention'))
    print('manager_name:', msg.get('manager_name'))
    print('-' * 60)

# Логическая проверка has_court_mention
print('\nПроверка соответствия has_court_mention и find_court_mentions(msg_text):')
mismatches_court = 0

for msg in valid_messages:
    text = msg.get('msg_text', '')
    expected = find_court_mentions(text)
    actual = msg.get('has_court_mention')
    if expected != actual:
        mismatches_court += 1

print('Всего несоответствий:', mismatches_court)

# Логическая проверка manager_name
print('\nПроверка соответствия manager_name и extract_manager_name(msg_text):')
mismatches_names = 0

for msg in valid_messages:
    text = msg.get('msg_text', '')
    expected_name = extract_manager_name(text)
    actual_name = msg.get('manager_name')
    if expected_name != actual_name:
        mismatches_names += 1

print('Всего несоответствий по ФИО:', mismatches_names)

Всего валидных сообщений: 48

Количество сообщений, в которых отсутствуют новые поля:
amounts: 0
has_court_mention: 0
manager_name: 0

Первые 3 сообщения после обогащения:

case_number: А60-12345/2025
org_name: ООО "Альфа-Строй"
amounts: ['15 000 000 руб.']
has_court_mention: True
manager_name: Иванов Иван Иванович
------------------------------------------------------------
case_number: А60-56789/2025
org_name: ЗАО "Бета-Инвест"
amounts: ['500 тыс. руб.']
has_court_mention: True
manager_name: None
------------------------------------------------------------
case_number: А60-11111/2025
org_name: ООО "Гамма-Трейд"
amounts: ['42 млн руб.']
has_court_mention: False
manager_name: Петров Пётр Петрович
------------------------------------------------------------

Проверка соответствия has_court_mention и find_court_mentions(msg_text):
Всего несоответствий: 0

Проверка соответствия manager_name и extract_manager_name(msg_text):
Всего несоответствий по ФИО: 0


### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [12]:
# Ваш код

from collections import Counter

# Количество сообщений по типам
type_stats = Counter(msg['type'] for msg in valid_messages)

# Количество сообщений по регионам (пропуская None)
region_stats = Counter(
    msg['region']
    for msg in valid_messages
    if msg.get('region') is not None
)

# Топ-5 публикаторов по количеству сообщений
publisher_stats = Counter(msg['publisher_inn'] for msg in valid_messages)
top_5_publishers = publisher_stats.most_common(5)

# Таймлайн - сортировка по дате публикации
timeline = sorted(valid_messages, key=lambda m: m['date_published'])

print('Статистика по типам:')
for t, cnt in type_stats.items():
    print(f'  {t}: {cnt}')

print('\nСтатистика по регионам:')
for r, cnt in region_stats.items():
    print(f'  {r}: {cnt}')

print('\nТоп-5 публикаторов (ИНН, количество):')
for inn, cnt in top_5_publishers:
    print(f'  {inn}: {cnt}')

print('\nТаймлайн (все сообщения):')
for m in timeline:
    # m['date_published'] — datetime после validate_message + parse_date
    date_str = m['date_published'].strftime('%d.%m.%Y')
    print(f'{date_str} | {m["publisher_inn"]} | {m["case_number"]} | {m["type"]}')

Статистика по типам:
  Введение процедуры: 8
  Собрание кредиторов: 3
  Завершение процедуры: 6
  Признание банкротом: 3
  Продажа имущества: 6
  Требование кредитора: 4
  Оспаривание сделки: 3
  Подача заявления: 3
  Отстранение управляющего: 1
  Мировое соглашение: 3
  Субсидиарная ответственность: 2
  Передача дела: 3
  Назначение управляющего: 2
  Жалоба на управляющего: 1

Статистика по регионам:
  Москва: 25
  Свердловская область: 6
  Санкт-Петербург: 5
  Краснодарский край: 3
  Челябинская область: 3
  Ростовская область: 2
  Владимирская область: 1
  Московская область: 2

Топ-5 публикаторов (ИНН, количество):
  7701234567: 3
  7706567890: 3
  7702345678: 2
  6658123450: 2
  7810345612: 2

Таймлайн (все сообщения):
10.01.2025 | 6658123450 | А60-25814/2025 | Завершение процедуры
15.01.2025 | 7701234567 | А60-12345/2025 | Введение процедуры
15.01.2025 | 5027123456 | А60-88888/2025 | Подача заявления
18.01.2025 | 7707345679 | А60-33333/2025 | Субсидиарная ответственность
22.01.20

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [13]:
# Ваш код

def prepare_message_for_json(msg):
    data = dict(msg)
    dt = data.get('date_published')
    if isinstance(dt, datetime):
        data['date_published'] = dt.strftime("%Y-%m-%d")
    return data

# Создаем новый список
valid_messages_for_json = [prepare_message_for_json(m) for m in valid_messages]

# Проверка работы кода 
print('Всего валидных сообщений:', len(valid_messages))
print('Всего сообщений после подготовки к JSON:', len(valid_messages_for_json))

# Посмотрим первые 3 записи до и после преобразования даты
print('\nСравнение первых 3 сообщений (тип поля date_published):\n')

for original, prepared in zip(valid_messages[:3], valid_messages_for_json[:3]):
    print('case_number:', original.get('case_number'))
    print('  Оригинал:', original.get('date_published'),
          '| тип:', type(original.get('date_published')))
    print('  Для JSON:', prepared.get('date_published'),
          '| тип:', type(prepared.get('date_published')))
    print('-' * 60)

Всего валидных сообщений: 48
Всего сообщений после подготовки к JSON: 48

Сравнение первых 3 сообщений (тип поля date_published):

case_number: А60-12345/2025
  Оригинал: 2025-01-15 00:00:00 | тип: <class 'datetime.datetime'>
  Для JSON: 2025-01-15 | тип: <class 'str'>
------------------------------------------------------------
case_number: А60-56789/2025
  Оригинал: 2025-02-10 00:00:00 | тип: <class 'datetime.datetime'>
  Для JSON: 2025-02-10 | тип: <class 'str'>
------------------------------------------------------------
case_number: А60-11111/2025
  Оригинал: 2025-03-15 00:00:00 | тип: <class 'datetime.datetime'>
  Для JSON: 2025-03-15 | тип: <class 'str'>
------------------------------------------------------------


### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [14]:
# Ваш код

# Сообщения по приоритетным делам (берем только из валидных)
priority_messages = [
    prepare_message_for_json(m)
    for m in valid_messages
    if m.get('case_number') in priority_set
]

analysis_results = {
    'valid_messages': valid_messages_for_json,
    'type_stats': dict(type_stats),
    'region_stats': dict(region_stats),
    'priority_messages': priority_messages,
}

with open('analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=4)

# Проверка работы кода    
import json
from pathlib import Path

print('Проверка файла analysis_results.json\n')

file_path = Path('analysis_results.json')

# Проверяем, что файл существует
if not file_path.exists():
    print('Файл analysis_results.json не найден.')
else:
    print('Файл найден.')
    print('Размер файла (байт):', file_path.stat().st_size)

    # Загружаем файл
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Проверяем ключи верхнего уровня
    print('\nКлючи верхнего уровня:')
    print(list(data.keys()))

    # Проверяем количества записей
    print('\nСравнение количества записей:')
    print('В valid_messages_for_json:', len(valid_messages_for_json))
    print('В файле valid_messages:', len(data.get('valid_messages', [])))

    print('\nВ priority_messages:', len(priority_messages))
    print('В файле priority_messages:', len(data.get('priority_messages', [])))

    # Проверяем статистики
    print('\nПроверка статистик:')
    print('type_stats в памяти:', dict(type_stats))
    print('type_stats в файле:', data.get('type_stats', {}))

    print('\nregion_stats в памяти:', dict(region_stats))
    print('region_stats в файле:', data.get('region_stats', {}))

    # Выводим первую запись для примера из valid_messages
    if data.get('valid_messages'):
        first = data['valid_messages'][0]
        print('\nПример первой записи из valid_messages:')
        print('case_number:', first.get('case_number'))
        print('date_published:', first.get('date_published'))
        print('Тип date_published:', type(first.get('date_published')))
        print('amounts:', first.get('amounts'))
        print('has_court_mention:', first.get('has_court_mention'))
        print('manager_name:', first.get('manager_name'))

    # Выводим первую запись для примера из priority_messages
    if data.get('priority_messages'):
        first_priority = data['priority_messages'][0]
        print('\nПример первой записи из priority_messages:')
        print('case_number:', first_priority.get('case_number'))
        print('date_published:', first_priority.get('date_published'))
        print('org_name:', first_priority.get('org_name'))

Проверка файла analysis_results.json

Файл найден.
Размер файла (байт): 59237

Ключи верхнего уровня:
['valid_messages', 'type_stats', 'region_stats', 'priority_messages']

Сравнение количества записей:
В valid_messages_for_json: 48
В файле valid_messages: 48

В priority_messages: 32
В файле priority_messages: 32

Проверка статистик:
type_stats в памяти: {'Введение процедуры': 8, 'Собрание кредиторов': 3, 'Завершение процедуры': 6, 'Признание банкротом': 3, 'Продажа имущества': 6, 'Требование кредитора': 4, 'Оспаривание сделки': 3, 'Подача заявления': 3, 'Отстранение управляющего': 1, 'Мировое соглашение': 3, 'Субсидиарная ответственность': 2, 'Передача дела': 3, 'Назначение управляющего': 2, 'Жалоба на управляющего': 1}
type_stats в файле: {'Введение процедуры': 8, 'Собрание кредиторов': 3, 'Завершение процедуры': 6, 'Признание банкротом': 3, 'Продажа имущества': 6, 'Требование кредитора': 4, 'Оспаривание сделки': 3, 'Подача заявления': 3, 'Отстранение управляющего': 1, 'Мировое согла

### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [15]:
# Ваш код
validation_results = {
    'error_records': error_records
}

with open('validation_errors.json', 'w', encoding='utf-8') as f:
    json.dump(validation_results, f, ensure_ascii=False, indent=4)

# Проверка работы кода
import json
from pathlib import Path

print('Проверка файла validation_errors.json\n')

file_path = Path('validation_errors.json')

# Проверяем, что файл существует
if not file_path.exists():
    print('Файл validation_errors.json не найден.')
else:
    print('Файл найден.')
    print('Размер файла (байт):', file_path.stat().st_size)

    # Читаем содержимое файла
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print('\nТип корневого объекта в файле:', type(data))

    # Извлекаем список error_records
    error_records_from_file = data.get("error_records", [])

    print('Количество error_records в памяти:', len(error_records))
    print('Количество error_records в файле:', len(error_records_from_file))

    # Выводим первую ошибочную запись для примера
    if error_records_from_file:
        first = error_records_from_file[0]
        print('\nПример первой ошибочной записи из файла:')
        print('Исходное сообщение:')
        print(first.get('message'))
        print('\nОшибки:')
        print(first.get('errors'))

Проверка файла validation_errors.json

Файл найден.
Размер файла (байт): 4193

Тип корневого объекта в файле: <class 'dict'>
Количество error_records в памяти: 6
Количество error_records в файле: 6

Пример первой ошибочной записи из файла:
Исходное сообщение:
{'publisher_inn': '', 'msg_text': 'Сообщение без ИНН публикатора. Тестовая запись.', 'date_published': '2025-04-25', 'type': 'Подача заявления', 'case_number': 'А60-24680/2025', 'org_name': None, 'region': None}

Ошибки:
[{'type': 'KeyError', 'message': "Отсутствует обязательное поле 'publisher_inn'"}]


### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [16]:
# Ваш код

# Собираем уникальные имена арбитражных управляющих из валидных сообщений
raw_manager_names = [
    m['manager_name']
    for m in valid_messages
    if m.get('manager_name')
]

# Очищаем: убираем точки в конце и лишние пробелы
manager_names = sorted(
    {name.rstrip(' .').strip() for name in raw_manager_names}
)

total_messages = len(messages)
total_valid = len(valid_messages)
total_errors = len(error_records)

from datetime import date
today_str = date.today().strftime('%d.%m.%Y')

lines = []

lines.append(f'АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО СООБЩЕНИЯМ ЕФРСБ')
lines.append(f'Дата формирования: {today_str}')
lines.append('')
lines.append(f'Всего сообщений: {total_messages}')
lines.append(f'Валидных сообщений: {total_valid}')
lines.append(f'Сообщений с ошибками: {total_errors}')
lines.append('')

lines.append('Статистика по типам сообщений:')
for t, cnt in type_stats.items():
    lines.append(f'  - {t}: {cnt}')
lines.append('')

lines.append('Статистика по регионам:')
for r, cnt in region_stats.items():
    lines.append(f'  - {r}: {cnt}')
lines.append('')

lines.append('Топ-5 публикаторов (по количеству сообщений):')
for inn, cnt in top_5_publishers:
    org_name = org_by_inn.get(inn, {}).get('name', 'Неизвестная организация')
    lines.append(f'  - {inn} ({org_name}): {cnt}')
lines.append('')

lines.append('Приоритетные дела (из перечня priority_cases):')
for cn in sorted(intersection_cases):
    lines.append(f'  - {cn}')
lines.append('')

lines.append('Список арбитражных управляющих, упомянутых в сообщениях:')
for fio in manager_names:
    lines.append(f'  - {fio}')

report_text = '\n'.join(lines)

with open('summary_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_text)

with open('summary_report.txt', 'r', encoding='utf-8') as f:
    print(f.read())

АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО СООБЩЕНИЯМ ЕФРСБ
Дата формирования: 24.04.2026

Всего сообщений: 54
Валидных сообщений: 48
Сообщений с ошибками: 6

Статистика по типам сообщений:
  - Введение процедуры: 8
  - Собрание кредиторов: 3
  - Завершение процедуры: 6
  - Признание банкротом: 3
  - Продажа имущества: 6
  - Требование кредитора: 4
  - Оспаривание сделки: 3
  - Подача заявления: 3
  - Отстранение управляющего: 1
  - Мировое соглашение: 3
  - Субсидиарная ответственность: 2
  - Передача дела: 3
  - Назначение управляющего: 2
  - Жалоба на управляющего: 1

Статистика по регионам:
  - Москва: 25
  - Свердловская область: 6
  - Санкт-Петербург: 5
  - Краснодарский край: 3
  - Челябинская область: 3
  - Ростовская область: 2
  - Владимирская область: 1
  - Московская область: 2

Топ-5 публикаторов (по количеству сообщений):
  - 7701234567 (ООО "Альфа-Строй"): 3
  - 7706567890 (ООО "Тета-Консалтинг"): 3
  - 7702345678 (ЗАО "Бета-Инвест"): 2
  - 6658123450 (ООО "Гамма-Трейд"): 2
  - 7810345612 (

---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |

